# Derivation, replay, and fair memory comparison

This lab focuses on the M3.4 distinction between **experience observed** and **experience declared as causal**. It then checks that derivation survives replay and that runs with different memory strategies remain comparable.

In [1]:
from pathlib import Path
import runpy

from meta_evolve import serialization
from meta_evolve.application import ExperienceQuery, replay_experience_graph

path = Path("main.py")
if not path.exists():
    path = Path("examples/03_experience_reuse/main.py")
demo = runpy.run_path(path)
results = demo["run_experiment"]()
print("comparison lab ready")

comparison lab ready


## 1. See where reuse changes the outcome

All variants spend the same task budget. Only the three memory-aware runs can turn the two failed evaluations into the successful value `10`.

In [2]:
for name, (result, _) in results.items():
    outcomes = tuple(
        trial.failure.kind
        if trial.failure is not None
        else trial.metrics["score"]
        for trial in result.trials()
    )
    print(
        f"{name:<10}",
        f"outcomes={outcomes}",
        f"task_usage={result.summary().usage}",
    )

no-memory  outcomes=(0, 'evaluator_failure', 'evaluator_failure', 'evaluator_failure') task_usage=Usage(evaluations=4, trials=3, tokens=0, wall_seconds=0.0)
push-only  outcomes=(0, 'evaluator_failure', 'evaluator_failure', 10) task_usage=Usage(evaluations=4, trials=3, tokens=0, wall_seconds=0.0)
pull-only  outcomes=(0, 'evaluator_failure', 'evaluator_failure', 10) task_usage=Usage(evaluations=4, trials=3, tokens=0, wall_seconds=0.0)
combined   outcomes=(0, 'evaluator_failure', 'evaluator_failure', 10) task_usage=Usage(evaluations=4, trials=3, tokens=0, wall_seconds=0.0)


## 2. Inspect exact committed derivation

A failed evaluation may still carry useful derivation. The second ordinary proposal cites one failure; the third cites two failures and, for pull variants, the compared trial occurrences.

In [3]:
for name in ("push-only", "pull-only", "combined"):
    result, _ = results[name]
    projection = result._projection()
    print(name)
    for submitted in projection.proposals:
        trial = projection.completed_trial(submitted.proposal.trial_id).trial
        kinds = tuple(ref.kind for ref in submitted.proposal.derived_from)
        print(f"  step={trial.logical_step} derived_from={kinds}")

push-only
  step=1 derived_from=()
  step=2 derived_from=('evaluation',)
  step=3 derived_from=('evaluation', 'evaluation')
pull-only
  step=1 derived_from=()
  step=2 derived_from=('evaluation',)
  step=3 derived_from=('evaluation', 'evaluation', 'trial', 'trial')
combined
  step=1 derived_from=()
  step=2 derived_from=('evaluation',)
  step=3 derived_from=('evaluation', 'evaluation', 'trial', 'trial')


Coordinator failure precedence is deliberate: resource exhaustion wins over invalid derivation, invalid derivation wins over a reported proposer failure, and either coordinator-imposed failure clears the edge and commits no candidate. A legitimate reported failure may retain valid derivation.

## 3. Replay the same edge graph

Replay validates the event stream and rebuilds derivation from durable proposals. It does not call the live context selector, reader, filtering code, or renderer.

In [4]:
result, storage = results["combined"]
events = storage.events.read(result.id)
left = replay_experience_graph(result.id, events)
right = replay_experience_graph(result.id, tuple(events))
query = ExperienceQuery(task_id=left.task_id, run_id=result.id)

left_edges = tuple(node.derived_from for node in left.state_at(query).nodes)
right_edges = tuple(node.derived_from for node in right.state_at(query).nodes)
print("replayed derivation equal:", left_edges == right_edges)
print("edge sizes:", tuple(len(edge) for edge in left_edges))

replayed derivation equal: True
edge sizes: (0, 0, 1, 4)


## 4. See the serialization compatibility rule

Empty derivation is omitted, so old proposal and event bytes remain unchanged. Non-empty derivation is encoded normally. An explicitly encoded empty default is rejected as non-canonical.

In [5]:
projection = result._projection()
bootstrap = projection.bootstrap.proposal
derived = projection.proposals[-1].proposal
bootstrap_bytes = serialization.dumps(bootstrap)
derived_bytes = serialization.dumps(derived)

print("empty derivation omitted:", "derived_from" not in bootstrap_bytes)
print("non-empty derivation encoded:", "derived_from" in derived_bytes)
print("round trip exact:", serialization.loads(derived_bytes) == derived)

empty derivation omitted: True
non-empty derivation encoded: True
round trip exact: True


## 5. Compare different memory strategies fairly

Memory declarations may differ. Compatibility still requires equal evaluator, proposer, objective, task/run budgets, artifact declaration, and initial digest. The comparison reports memory-declaration equality rather than rejecting the pair.

In [6]:
baseline = results["no-memory"][0]
for name, (other, _) in results.items():
    comparison = baseline.compare(other)
    winner = comparison.winner.primary_score if comparison.winner else "tie"
    print(
        f"no-memory vs {name:<10}",
        f"winner={winner}",
        f"same_seed={comparison.same_random_seed}",
        f"same_context={comparison.same_context_declaration}",
        f"same_pull={comparison.same_experience_declaration}",
    )

no-memory vs no-memory  winner=tie same_seed=True same_context=True same_pull=True
no-memory vs push-only  winner=10 same_seed=True same_context=False same_pull=True
no-memory vs pull-only  winner=10 same_seed=True same_context=True same_pull=False
no-memory vs combined   winner=10 same_seed=True same_context=False same_pull=False


## Takeaway

M3.4 does not claim that every observed record caused a result. It preserves the proposer's explicit usage intent while the coordinator independently enforces visibility, ordering, budgets, failure semantics, and evaluation authority.